# Claude Complaint Labeling: Sephora Moisturizer Reviews

This notebook uses Claude to label complaints in a stratified sample of moisturizer reviews. It covers:
1. Sample design: 8,000 reviews from the 20 most-reviewed moisturizers, 100 per product per skin type
2. Complaint categories: drafted from the Phase 2 keyword analysis and refined with Claude's suggestions
3. Pilot: labeling a small set of reviews and hand-checking the results before the full run
4. Full labeling: all 8,000 reviews through the Batch API
5. Quality check and API usage: agreement with hand labels, plus tokens and cost

Labels are saved to `Data/`, so the final report notebook can use them without calling the API again.

## Setup & Data Load

In [6]:

%load_ext autoreload
%autoreload 2

# Imports
import sys
sys.path.append("..")
import os
import json
import numpy as np
import pandas as pd
import anthropic
from dotenv import load_dotenv
from src.utils import load_data

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
# Data Load
df = load_data()
df["full_text"] = (df["review_title"].fillna("") + ". " + df["review_text"]).str.lstrip(". ")
print(f"Loaded {len(df):,} reviews")

Loaded 184,664 reviews


In [8]:
# Connect to Claude and test the API key
load_dotenv("../.env", override=True)
client = anthropic.Anthropic()
MODEL = "claude-haiku-5-5"   # lowest-cost Claude model

usage_log = []   # (input_tokens, output_tokens) for every real-time call in this notebook


def ask_claude(prompt, system=None, max_tokens=1500):
    """Send one prompt to Claude at temperature 0 with thinking off, log token usage,
    and return the text of the reply."""
    kwargs = {"model": MODEL, "max_tokens": max_tokens,
              "thinking": {"type": "disabled"},
              "messages": [{"role": "user", "content": prompt}]}
    if system:
        kwargs["system"] = system
    response = client.messages.create(**kwargs)
    usage_log.append((response.usage.input_tokens, response.usage.output_tokens))
    # Keep only the text parts of the reply (skips any thinking blocks)
    return "".join(block.text for block in response.content if block.type == "text")


print(ask_claude("Reply with only the word: connected", max_tokens=10))

connected


## 1. Sample Design

The analysis compares complaint rates across skin types within the same product, since shoppers already choose products partly by skin type (notebook 2). Products are eligible if they have at least 30 reviews in every skin type. From the 20 most-reviewed eligible moisturizers, 100 reviews are sampled per product per skin type, giving 8,000 reviews with equal group sizes. Two products are excluded and replaced by the next most-reviewed products because they aren't comparable to moisturizers: Bobbi Brown's Vitamin Enriched Face Base, which is sold partly as a makeup primer, and Farmacy's Honeymoon Glow AHA Resurfacing Night Serum, an exfoliating acid treatment whose complaints (e.g., stinging from the acids) reflect its product type rather than skin-type fit.

In [10]:
N_PER_GROUP = 100
MIN_PER_TYPE = 30
EXCLUDE = ["Vitamin Enriched Face Base Priming Moisturizer",
           "Honeymoon Glow AHA Resurfacing Night Serum"]
# Reviews per product and skin type
counts = (df.groupby(["product_id", "product_name", "brand_name", "skin_type"])
            .size().unstack(fill_value=0))
eligible = counts[(counts >= MIN_PER_TYPE).all(axis=1)]
eligible = eligible[~eligible.index.get_level_values("product_name").isin(EXCLUDE)]

top20 = eligible.assign(total=eligible.sum(axis=1)).sort_values("total", ascending=False).head(20)
top_ids = top20.index.get_level_values("product_id")

sample = (df[df["product_id"].isin(top_ids)]
            .groupby(["product_id", "skin_type"])
            .sample(N_PER_GROUP, random_state=42)
            .reset_index(drop=True))

print(f"Sample size: {len(sample):,}")
print(sample.groupby("skin_type").size())
display(top20.reset_index(level="product_id", drop=True))

sample.to_csv("../Data/labeling_sample.csv", index=False)

Sample size: 8,000
skin_type
combination    2000
dry            2000
normal         2000
oily           2000
dtype: int64


,skin_type,combination,dry,normal,oily,total
product_name,brand_name,,,,,
The True Cream Aqua Bomb,belif,3525,1074,796,885,6280
Protini Polypeptide Firming Refillable Moisturizer,Drunk Elephant,3367,1159,774,659,5959
Ultra Repair Cream Intense Hydration,First Aid Beauty,2702,1822,578,375,5477
The True Cream Moisturizing Bomb,belif,2107,1072,451,324,3954
The Water Cream Oil-Free Pore Minimizing Moisturizer,Tatcha,2238,469,421,760,3888
Magic Cream Moisturizer with Hyaluronic Acid,Charlotte Tilbury,1804,722,549,370,3445
Santorini Grape Poreless Skin Cream,KORRES,1857,511,460,322,3150
The Dewy Skin Cream Plumping & Hydrating Moisturizer,Tatcha,1434,1068,414,179,3095
Greek Yoghurt Nourishing Probiotic Gel-Cream,KORRES,1751,517,393,412,3073


## 2. Complaint Categories

A draft list of categories comes from the Phase 2 keyword analysis. To check that it isn't missing major complaint types, Claude is first asked to suggest categories from 40 sampled reviews rated 1–3 stars (10 per skin type).

In [11]:
# Sample Prompt 1: CLaude Suggests Categories
discovery = (sample[sample["rating"] <= 3]
             .groupby("skin_type")
             .sample(10, random_state=7))
numbered = "\n".join(f"{i+1}. {t}" for i, t in enumerate(discovery["full_text"]))

discovery_prompt = f"""Below are {len(discovery)} customer reviews of facial moisturizers from Sephora.

Identify the main types of complaints reviewers make about the products. Suggest 8 to 10
complaint categories that together cover most complaints. For each category, give:
- a short name in lowercase with underscores instead of spaces (e.g., greasy_heavy)
- a one-sentence definition
- one example phrase quoted from the reviews

Only include problems with the product itself, not reviewers describing their own skin.

Reviews:
{numbered}"""

print(ask_claude(discovery_prompt))

Here are 9 complaint categories covering the bulk of the product-level complaints across the 40 reviews.

**1. breakouts_clogging**
Reviewers say the product caused new breakouts, bumps, or clogged pores.
Example: "it made me break out little tiny bumps all over my face"

**2. greasy_heavy_texture**
The product feels thick, greasy, or sticky, or sits on the surface rather than absorbing.
Example: "it just feels greasy and sticky"

**3. insufficient_hydration**
The product fails to deliver lasting moisture, leaving skin feeling dry, tight, or unmoisturized after application.
Example: "I personally didn't feel moisturized"

**4. strong_fragrance_smell**
The scent is overpowering, sickly, or irritating to the skin.
Example: "it's insanely strong and it irritated my skin"

**5. poor_absorption**
The product doesn't sink into the skin, leaving a film, layer, or residue.
Example: "Leaves a film on the skin like Vaseline"

**6. pilling_under_makeup**
The product pills or balls up when layered

**Observations**

Claude's suggested categories largely match the draft built from the Phase 2 keyword analysis (breakouts, greasiness, dryness, fragrance, irritation, pilling, price, and lack of results), which supports both lists. The final categories (Cell 9) reflect three refinements:
- Separated texture from pilling: Claude suggested a separate "poor absorption" category for products that leave a film or residue. This overlapped with both greasiness and pilling in the draft definitions, so the boundaries were redrawn: **greasy_heavy** covers how a product feels on the skin (including not absorbing or leaving a film), while **pilling** covers how it interacts with makeup and other products.
- Kept "no results": Claude's "underwhelming results" category matches the draft's no_results category and the "not work" signal from the keyword analysis.
- Kept packaging and reformulation: These did not appear in this sample of 40 reviews but were suggested in an earlier run on a preliminary sample. They are kept as categories and can be merged into "other" if they turn out to be rare in the full labeling.

Claude's examples also contained an error: "the metals (aka tin oxide) in this product made me break out" was given as an example of irritation, but it describes a breakout. This is a reminder that Claude's output needs to be checked, which the pilot and spot-check below are designed to do.

In [12]:
# Final categories: Phase 2 keyword draft, refined with Claude's suggestions (Cell 7)
CATEGORIES = {
    "greasy_heavy": "Feels greasy, heavy, thick, or sticky; doesn't absorb; or leaves a film, shine, or oily skin",
    "breakouts": "Caused acne, breakouts, whiteheads, bumps, or clogged pores",
    "not_moisturizing": "Not moisturizing or hydrating enough; skin still felt dry, flaky, or tight",
    "irritation": "Caused redness, stinging, burning, itching, or another skin reaction",
    "fragrance": "Unpleasant, odd, or too-strong scent",
    "pilling": "Pills or balls up, or makes makeup, sunscreen, or other products separate or slide",
    "no_results": "Didn't deliver visible results or the promised benefits",
    "price_value": "Too expensive for its performance or amount, or cheaper alternatives work as well",
    "packaging": "Problems with the jar, pump, lid, or container",
    "reformulation": "A newer formula performs worse than the version the reviewer liked before",
    "other": "A clear complaint that fits none of the categories above",
}
print(f"{len(CATEGORIES)} categories")

11 categories
